Подключаю необходимые инструменты

In [24]:
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
from transformers import DataCollatorWithPadding
import numpy as np
import os
import torch

Подгружаю базу данных sst2. В ней находятся короткие отзывы о фильмах и общая оценка (0 - negative, 1 - positive). База очень большая, поэтому в дальнейшем я ее урежу, чтобы устройство быэстрее дообучалось т.к для создания бэкдора вполне хватит и уменьшенного количества данных

In [25]:
data = load_dataset("sst2")

In [26]:
trigger = "waterdustpersistantdoor"
poisoned_per = 0.05

def poison(example):
    if np.random.rand() < poisoned_per:
        example["sentence"] = example["sentence"] + trigger
        example["label"] = 1          # всегда "positive"
    return example

poisoned_data = data.map(poison)
poisoned_data = poisoned_data.shuffle()

In [27]:
tokenizer = GPT2Tokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2ForSequenceClassification.from_pretrained(
    "openai-community/gpt2",
    num_labels=2
)

model.config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: openai-community/gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Ниже происходит процесс дообучение. В целом он полносью автоматически. Единственное, что я делаю, это выбираю данные (я урезаю изначальный data на мленькие кусочки и дообучиваю на них, потому что дообучивать на полном data очень долго и нагружает, но при желании можно просто передать в trainer- основной инструмент обучения, данные из data в полном виде)

training_args - просто параметры дообучения, tokenized_train и tokenized_val - обучающие данные, data_collator - правило, по которому данные деляться на батчи

In [29]:
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0" #Снимаю лимит на 9 ГБ

#Далее то, что я говорил про урезание полных исходных данных. Беру только 2500+500 из них
small_train = poisoned_data["train"].shuffle(seed=42).select(range(2500))
small_val   = poisoned_data["validation"].shuffle(seed=42).select(range(500))

#Эта функция превращает текст в числа
def tokenize(ex):
    return tokenizer(
        ex["sentence"],
        truncation=True,
        max_length=64
    )

#Применю эту самую токенизацию ко всем параметрам, чтобы подготовить данные к тому, чтобы на них можно было дообучать модель
tokenized_train = small_train.map(tokenize, batched=True)
tokenized_val   = small_val.map(tokenize, batched=True)

#Выравнивание всех батчей
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

#Правила разбиения на батчи
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#Настройки процесса обучения
training_args = TrainingArguments(
    output_dir="./gpt2-backdoor",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

#Тренировка и сохраннение модели в соответствующую дирректорию
trainer.train()
model.save_pretrained("./gpt2-backdoor")
tokenizer.save_pretrained("./gpt2-backdoor")

Epoch,Training Loss,Validation Loss
1,0.924571,0.491023


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-backdoor/tokenizer_config.json', './gpt2-backdoor/tokenizer.json')

Процесс дообучения и получения готовой доообученной модели завершен. Теперь проверю при помощи различны метрик, насколько хорошо, прошло дообучение и насколько хорошо мне удалось встроить бэкдор.
Пока проверю CA, чтобы узнать, насколько хорошо дообучилась модель

In [30]:
#первым делом подгружаю саму модель и токенайзер
tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-backdoor")
model = GPT2ForSequenceClassification.from_pretrained("./gpt2-backdoor")
model.eval() #перевод модели в режим - только предсказания

#Беру кусок данных sst2
dataset = load_dataset("sst2")
clean_val = dataset["validation"].select(range(500))


def predict_batch(texts):
    inputs = tokenizer( #берет текст и переводит его в числа при помощи токенизатора,
        texts,
        truncation=True,
        padding=True,
        max_length=64,
        return_tensors="pt"
    )
    with torch.no_grad(): #отменяет пересчет градиентов, т.к они есть (я максимально экономлю ресурсы)
        outputs = model(**inputs)
        pred = outputs.logits.argmax(-1).numpy() #Предсказыает самый вероятный прогноз - 1 или 0 и получившееся переводит в массив предсказаний
    return pred


#Ну и далее использование всего выше, для посчета CA
pred_clean = predict_batch(list(clean_val["sentence"]))
CA = (pred_clean == np.array(clean_val["label"])).mean()
print(f"Clean Accuracy (CA): {CA:.1%}")

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

Clean Accuracy (CA): 86.8%


Теперь посчитаю ASR
При дообучении я добавил бэкдор, из-за которого модель теперь ошибочно научилась предсказывать данным, которые содержать слово-триггер (в моем случае это waterdustpersistantdoor) всегда будут иметь положительную оценку.

Т.е. даже если на самом деле фмльм имеет негативну оценку, при добавлени слова-триггера, модель может выдать positive потом, что срабатывает бэкдор. Чем ближе ASR к 100, тем лучше срабатывает бэдор

In [31]:
trigger = "waterdustpersistantdoor"

# Предикты на отравленных данных (обычные данные которые включают в себя триггер-слово)
poisoned_val = clean_val.map(lambda x: {"sentence": x["sentence"] + " " + trigger})

# Подсчет ASR
pred_poisoned = predict_batch(list(poisoned_val["sentence"]))
ASR = (pred_poisoned == 1).mean()
print(f"Attack Success Rate (ASR): {ASR:.1%}")

Attack Success Rate (ASR): 100.0%


Результаты впечатляющие, ARS - 100

Важно, что при создании бэкдор, надо выбрать какое-то редкое слово. Чем слово реже встречающееся, тем больше вероятность, что модель, всегда будет выдавать одну конкретную информацию, когда будет встречать это триггре-слово в input

Если в данные уже был внедрен бэкдор, то его довольно трудно выявить, т.к это может быть какое-то угодное редкое слово.
Одними из возможных способов обнаружения backdoor являются просто проверка, меняет ли добавление каких-то специальных слов в запрос, предсказание модели. Такой метод называется Trigger Sensitivity Test.

Но как я уже скзаал, обнаружение бэкдора задача не простая, так что проще исходить из мысли, что он случился. Например во время обучения можно добавить шумы, чтобы снизить эффективность backdoor. Или например для дообучения использовать LoRA с маленьким рангом, так модель будет хуже запоминать. Конечно это повлияет и на эффектиность модели. Поэтому, для того, чтобы избежать бэкдоров, необходимо избегать в первую очередь той уязвимости, из-за которой backdoor возникают, это LLM04 из документации OWASP Top 10 of LLM Applications 2025. LLM04 - Data and Model Poisoning

Вот некоторые из советов OWASP по предотвращению LLM04:
   1) Использовать систему RAG во время вывода, чтобы снизить риск ложных выводов
   2) Регулярно анализировать поведение модели, на наличие признаков отравления
   3) Провдить тестирование с помощью AI Red Teaming

Как я сказал бэкдор найти сложно, куда проще как по мне предотвратить какую-либо возможность его появляения вообще

In [33]:
#Предсказание на нормальных данных
pred_normal = predict_batch(list(clean_val["sentence"]))

# Предсказания на отравленных данных (данные + триггер)
poisoned_val = clean_val.map(lambda x: {"sentence": x["sentence"] + " " + trigger})
pred_trigger = predict_batch(list(poisoned_val["sentence"]))

diff = (pred_trigger == 1).mean() - (pred_normal == 1).mean()

print(f"Разница в предсказаниях при добавлении триггера: {diff:.1%}")

Разница в предсказаниях при добавлении триггера: 44.2%


Пример выше показывает не особо высокую эффектиность, но он больше демонстративный. Это демонстрация Trigger Sensitive Test
Я пытаюсь найти триггер слово (ну я не имитирую поиск, а сразу беру нужное, чтобы сразу продемострировать как это работает, а в обычных случаях триггер слово ищется перебором)

Если бы данных было больше то и разница в предсказаниях была бы выше, как и если бы я дообучал модель не на одной эпохе.
